# FASE 2. EXPLORACIÓN, PREPROCESAMIENTO Y VALIDACIÓN DE DATOS

**Proyecto:** Análisis del impacto operacional documentado de las fallas del Sistema Eléctrico Nacional sobre comunas prioritarias de Chile mediante ciencia de datos.

**Asignatura:** Programación para la Ciencia de Datos (MCDI500)
**Programa:** Magíster en Ciencia de Datos e Inteligencia Artificial
**Institución:** Universidad Andrés Bello
**Grupo:** 7

---

## 1. Introducción y objetivos de la Fase 2

### 1.1. Introducción

La Fase 2 desarrolla el procesamiento inicial de los registros históricos de fallas del Sistema Eléctrico Nacional (SEN), con el propósito de establecer una base de datos consistente, reproducible y técnicamente adecuada para el análisis territorial.

El conjunto de información proviene de las publicaciones del Coordinador Eléctrico Nacional (CEN), correspondientes al período enero de 2024 a septiembre de 2026. Durante la Fase 1 se implementó un procedimiento de consolidación que permitió integrar 33 publicaciones mensuales, obteniendo un conjunto preliminar de 18.019 registros y 40 variables.

Sin embargo, la disponibilidad de los registros consolidados no garantiza directamente su utilización analítica. Es necesario examinar la integridad de las variables, identificar inconsistencias, evaluar los valores faltantes y establecer reglas de transformación coherentes con las características de los eventos documentados.

El nuevo enfoque territorial requiere especial atención a las variables `Comuna`, `Zona Afectada`, `Consumo` y `Distribuidoras Afectadas`, así como a la identificación de informes, sus estados administrativos y fechas relevantes.

Adicionalmente, se contempla incorporar información territorial oficial del Instituto Nacional de Estadísticas (INE), utilizando los códigos comunales y la población censada en 2024 como antecedentes para identificar las 15 comunas prioritarias del estudio.

Esta integración permitirá preparar una estructura de datos que relacione los informes de fallas con las comunas mencionadas, conservando la trazabilidad de los registros originales.

### 1.2. Objetivo general

**Preparar y validar un conjunto de datos integrado de fallas del Sistema Eléctrico Nacional y antecedentes territoriales oficiales, mediante técnicas de exploración, limpieza y transformación en Python, para sustentar el análisis comparativo de exposición operacional documentada de comunas prioritarias de Chile.**

### 1.3. Objetivos específicos

1. **Caracterizar** la estructura, composición y cobertura temporal de los registros consolidados del CEN mediante herramientas de exploración de datos.

2. **Diagnosticar** la calidad de la información, identificando valores faltantes, duplicados, inconsistencias y restricciones de las variables territoriales y operacionales.

3. **Normalizar** las variables relevantes para el análisis comunal, considerando la separación de nombres territoriales, las categorías de consumo y los tipos de datos.

4. **Integrar** los registros de fallas con información territorial oficial del INE, estableciendo correspondencias verificables entre comunas y códigos territoriales.

5. **Validar** la integridad, consistencia y trazabilidad del conjunto transformado mediante pruebas funcionales y controles de calidad.

### 1.4. Alcance metodológico

La Fase 2 comprende la obtención y lectura de los datos, su exploración inicial, el diagnóstico de calidad, la implementación de transformaciones justificadas y la validación del conjunto resultante.

Se conservará la información original y se documentarán las modificaciones realizadas durante el procesamiento.

La unidad documental será el informe de falla identificado mediante `id_falla`. Para el análisis territorial se construirá una segunda estructura que represente las asociaciones entre informes y comunas, sin interpretar cada asociación como un evento físico independiente.

Los indicadores de afectación operacional se construirán posteriormente sobre registros depurados, distinguiendo los eventos con afectación de consumo reportada de aquellos que no presentan dicha condición.

La información disponible no permite atribuir automáticamente interrupciones efectivas a todos los habitantes de una comuna ni calcular indicadores regulatorios como SAIDI y SAIFI sin antecedentes complementarios de clientes afectados y duración de sus interrupciones.

### 1.5. Criterios de reproducibilidad

El procesamiento se desarrollará mediante Python y las bibliotecas Pandas y NumPy, utilizando Jupyter Notebook como entorno de documentación y ejecución.

Las operaciones estarán organizadas en bloques funcionales, incorporando controles de entrada, validaciones de resultados y explicaciones técnicas de las transformaciones.

Los conjuntos de datos originales no serán sobrescritos. Los productos intermedios y finales se almacenarán en directorios diferenciados, manteniendo la posibilidad de reconstruir el procesamiento a partir de las fuentes documentadas.


In [3]:
# ==========================================================
# FASE 2 - OBTENCIÓN Y CARGA DEL DATASET
# PROYECTO MCDI500 - GRUPO 7
# ANÁLISIS TERRITORIAL DE FALLAS DEL SEN
# ==========================================================

from pathlib import Path

import pandas as pd
import numpy as np

from IPython.display import display


# ==========================================================
# 1. LOCALIZAR LA CARPETA PRINCIPAL DEL PROYECTO
# ==========================================================

def localizar_proyecto(ruta_inicial):

    """
    Busca la carpeta principal del proyecto
    a partir del directorio de ejecución.
    """

    inicio = Path(ruta_inicial).resolve()

    for carpeta in [inicio, *inicio.parents]:

        if (
            (carpeta / "F1").is_dir()
            and (carpeta / "src").is_dir()
            and (carpeta / "data").is_dir()
        ):

            return carpeta

    raise FileNotFoundError(
        "No se encontró la carpeta principal "
        "del proyecto MCDI500."
    )


RAIZ = localizar_proyecto(Path.cwd())


# ==========================================================
# 2. DEFINIR LA FUENTE DE DATOS
# ==========================================================

ARCHIVO_DATOS = (
    RAIZ
    / "data"
    / "processed"
    / "cen_fallas_2024_2026_eventos.csv"
)


# ==========================================================
# 3. VERIFICAR LA EXISTENCIA DEL ARCHIVO
# ==========================================================

if not ARCHIVO_DATOS.is_file():

    raise FileNotFoundError(
        f"No se encontró el dataset: {ARCHIVO_DATOS}"
    )


# ==========================================================
# 4. CARGAR LOS DATOS SIN TRANSFORMACIONES
# ==========================================================

df_cen = pd.read_csv(
    ARCHIVO_DATOS,
    encoding="utf-8-sig",
    dtype="string",
    low_memory=False
)


# ==========================================================
# 5. VERIFICAR LA ESTRUCTURA MÍNIMA
# ==========================================================

COLUMNAS_REQUERIDAS = [
    "Número",
    "id_falla",
    "Fecha Perturbación",
    "Estado",
    "Comuna",
    "Zona Afectada",
    "Consumo",
    "anio_publicacion",
    "mes_publicacion"
]

columnas_faltantes = sorted(
    set(COLUMNAS_REQUERIDAS) - set(df_cen.columns)
)

if columnas_faltantes:

    raise ValueError(
        "Faltan columnas necesarias para F2: "
        f"{columnas_faltantes}"
    )

if df_cen.empty:

    raise ValueError(
        "El conjunto de datos está vacío."
    )


# ==========================================================
# 6. MOSTRAR RESULTADOS DE CARGA
# ==========================================================

print("=" * 60)
print("FASE 2 - CARGA DEL CONJUNTO DE DATOS")
print("=" * 60)

print("\nCarpeta del proyecto:")
print(RAIZ)

print("\nArchivo utilizado:")
print(ARCHIVO_DATOS.name)

print("\nDimensiones del dataset:")
print("Registros:", df_cen.shape[0])
print("Variables:", df_cen.shape[1])

print("\nColumnas requeridas:")
print("[OK] Todas están presentes")

print("\nPrimeras observaciones:")

columnas_vista = [
    "Número",
    "Fecha Perturbación",
    "Estado",
    "Comuna",
    "Consumo",
    "Zona Afectada"
]

display(
    df_cen[columnas_vista].head(5)
)

print("\n[OK] Dataset cargado sin modificar los datos")

FASE 2 - CARGA DEL CONJUNTO DE DATOS

Carpeta del proyecto:
C:\Users\jeanp\OneDrive\00. MAGISTER CIA\M1_Programación para la Ciencia de Datos\Proyecto_MCDI500

Archivo utilizado:
cen_fallas_2024_2026_eventos.csv

Dimensiones del dataset:
Registros: 18019
Variables: 40

Columnas requeridas:
[OK] Todas están presentes

Primeras observaciones:


,Número,Fecha Perturbación,Estado,Comuna,Consumo,Zona Afectada
0,2024000001.0,2024-01-01 10:15:00,finalizado,San Clemente,consumo_no_afectado,Maule
1,2024000002.0,2024-01-01 08:20:00,finalizado,Huara,consumo_no_afectado,Tarapacá
2,2024000004.0,2024-01-01 08:20:00,finalizado,Camarones,consumo_no_afectado,Arica y Parinacota
3,2024000003.0,2024-01-01 08:20:00,finalizado,Camarones,consumo_no_afectado,Arica y Parinacota
4,2024000087.0,2024-01-09 23:30:00,finalizado,Los Andes,consumo_no_afectado,Valparaíso



[OK] Dataset cargado sin modificar los datos


In [4]:
# ==========================================================
# FASE 2 - EXPLORACIÓN INICIAL
# CARACTERIZACIÓN TERRITORIAL Y OPERACIONAL
# ==========================================================


# ==========================================================
# 1. DEFINIR VARIABLES DE INTERÉS
# ==========================================================

variables_interes = [
    "Número",
    "id_falla",
    "Comuna",
    "Zona Afectada",
    "Consumo",
    "Distribuidoras Afectadas",
    "Fecha Perturbación",
    "Fecha Efectiva Fin",
    "Estado",
    "Tipo",
    "Causa"
]

variables_disponibles = [
    columna
    for columna in variables_interes
    if columna in df_cen.columns
]


# ==========================================================
# 2. CALCULAR COMPLETITUD DE VARIABLES
# ==========================================================

diagnostico = pd.DataFrame({

    "Variable": variables_disponibles,

    "Registros completos": [
        int(df_cen[columna].notna().sum())
        for columna in variables_disponibles
    ],

    "Valores faltantes": [
        int(df_cen[columna].isna().sum())
        for columna in variables_disponibles
    ]

})

diagnostico["% faltantes"] = (
    diagnostico["Valores faltantes"]
    / len(df_cen)
    * 100
).round(2)


# ==========================================================
# 3. MOSTRAR COMPLETITUD
# ==========================================================

print("=" * 60)
print("DIAGNÓSTICO INICIAL DE VARIABLES")
print("=" * 60)

display(
    diagnostico.sort_values(
        "% faltantes",
        ascending=False
    )
)


# ==========================================================
# 4. DISTRIBUCIÓN DE CATEGORÍAS DE CONSUMO
# ==========================================================

print("\nDISTRIBUCIÓN DE LA VARIABLE CONSUMO")

display(
    df_cen["Consumo"]
    .value_counts(dropna=False)
    .rename_axis("Categoría")
    .reset_index(name="Registros")
)


# ==========================================================
# 5. IDENTIFICAR REGISTROS MULTICOMUNA
# ==========================================================

comunas_multiples = (
    df_cen["Comuna"]
    .fillna("")
    .str.contains("|", regex=False)
)

print("\nDIAGNÓSTICO TERRITORIAL")

print(
    "Registros con comuna informada:",
    int(df_cen["Comuna"].notna().sum())
)

print(
    "Registros con varias comunas:",
    int(comunas_multiples.sum())
)


# ==========================================================
# 6. VERIFICAR IDENTIFICADORES
# ==========================================================

print("\nCONTROL INICIAL DE IDENTIFICADORES")

print(
    "Identificadores faltantes:",
    int(df_cen["id_falla"].isna().sum())
)

print(
    "Identificadores repetidos:",
    int(
        df_cen["id_falla"].duplicated(keep=False).sum()
    )
)


# ==========================================================
# 7. RESULTADO DE EXPLORACIÓN
# ==========================================================

print("\n[OK] Exploración inicial finalizada")
print("[PENDIENTE] Diagnóstico exhaustivo y limpieza")

DIAGNÓSTICO INICIAL DE VARIABLES


,Variable,Registros completos,Valores faltantes,% faltantes
5,Distribuidoras Afectadas,6809,11210,62.21
7,Fecha Efectiva Fin,10805,7214,40.04
2,Comuna,16874,1145,6.35
3,Zona Afectada,16927,1092,6.06
4,Consumo,17732,287,1.59
1,id_falla,18019,0,0.00
0,Número,18019,0,0.00
6,Fecha Perturbación,18019,0,0.00
8,Estado,18019,0,0.00
9,Tipo,18019,0,0.00



DISTRIBUCIÓN DE LA VARIABLE CONSUMO


,Categoría,Registros
0,consumo_no_afectado,10901
1,consumo_regulado,3932
2,consumo_libre_regulado,1878
3,consumo_libre,1021
4,<NA>,287



DIAGNÓSTICO TERRITORIAL
Registros con comuna informada: 16874
Registros con varias comunas: 2623

CONTROL INICIAL DE IDENTIFICADORES
Identificadores faltantes: 0
Identificadores repetidos: 0

[OK] Exploración inicial finalizada
[PENDIENTE] Diagnóstico exhaustivo y limpieza


## 4. Diagnóstico de calidad de datos

### 4.1. Fundamentación técnica

La calidad de los datos constituye una condición fundamental para desarrollar análisis territoriales consistentes. La existencia de registros históricos consolidados no garantiza que todas sus variables presenten niveles adecuados de completitud, integridad y consistencia.

En esta etapa se realiza un diagnóstico sistemático del conjunto de datos del Coordinador Eléctrico Nacional, considerando las siguientes dimensiones:

**Completitud:** identificación de valores ausentes y campos vacíos, especialmente en las variables asociadas a comunas, afectación del consumo, fechas y estados administrativos.

**Unicidad:** verificación de identificadores y registros repetidos, considerando que el conjunto consolidado debe conservar una unidad documental claramente definida.

**Consistencia:** identificación de diferencias en la representación de categorías, espacios innecesarios y posibles valores no estandarizados.

**Integridad territorial:** evaluación de la disponibilidad de información comunal y reconocimiento de registros que mencionan varias comunas dentro de un mismo campo.

**Integridad temporal:** identificación de información de fechas incompleta, como antecedente para las verificaciones cronológicas posteriores.

### 4.2. Criterios de evaluación

El diagnóstico se desarrollará sobre el conjunto original cargado en memoria, sin eliminar registros ni modificar los valores existentes.

Se calcularán indicadores de completitud y se identificarán posibles inconsistencias que requieran tratamiento durante la preparación de los datos.

Los resultados permitirán establecer reglas diferenciadas de limpieza y transformación, considerando el significado técnico de cada variable.

En particular, la ausencia de información comunal no se interpretará automáticamente como inexistencia de afectación territorial. Del mismo modo, la falta de una fecha de término no permitirá asumir que el evento permanece activo ni asignar arbitrariamente una duración.

La identificación de problemas de calidad permitirá fundamentar las decisiones de preprocesamiento y conservar la trazabilidad entre los registros originales y los conjuntos transformados.


In [5]:
# ==========================================================
# FASE 2 - APARTADO 4
# DIAGNÓSTICO DE CALIDAD DE DATOS
# PROYECTO MCDI500 - GRUPO 7
# ==========================================================

import pandas as pd
from IPython.display import display


# ==========================================================
# 1. FUNCIÓN PARA EVALUAR LA CALIDAD DEL DATASET
# ==========================================================

def diagnosticar_calidad(df):
    """
    Evalúa la completitud y unicidad del dataset.

    No modifica los registros originales.
    """

    if df.empty:
        raise ValueError("El dataset está vacío.")

    resultados = []

    for columna in df.columns:

        serie = df[columna].astype("string")

        # Valores ausentes reconocidos por Pandas
        nulos = int(serie.isna().sum())

        # Cadenas vacías o compuestas solamente por espacios
        vacios = int(
            serie.str.strip()
            .eq("")
            .fillna(False)
            .sum()
        )

        faltantes_totales = nulos + vacios

        resultados.append({
            "Variable": columna,
            "Valores nulos": nulos,
            "Campos vacíos": vacios,
            "Total faltantes": faltantes_totales,
            "% faltantes": round(
                faltantes_totales / len(df) * 100, 2
            )
        })

    return pd.DataFrame(resultados)


# ==========================================================
# 2. EJECUTAR DIAGNÓSTICO
# ==========================================================

calidad_df = diagnosticar_calidad(df_cen)

print("=" * 65)
print("DIAGNÓSTICO GENERAL DE CALIDAD")
print("=" * 65)

print(f"\nRegistros analizados: {len(df_cen):,}")
print(f"Variables analizadas: {len(df_cen.columns)}")

print("\nVARIABLES CON MAYOR PORCENTAJE DE DATOS FALTANTES")

display(
    calidad_df.sort_values(
        "% faltantes",
        ascending=False
    ).head(15)
)


# ==========================================================
# 3. VERIFICAR IDENTIFICADORES
# ==========================================================

print("\n" + "=" * 65)
print("CONTROL DE IDENTIFICADORES")
print("=" * 65)

identificadores = df_cen["id_falla"].astype("string").str.strip()

id_validos = identificadores.notna() & identificadores.ne("").fillna(False)

duplicados_id = identificadores[
    id_validos
].duplicated(keep=False)

print(
    "\nIdentificadores ausentes o vacíos:",
    int((~id_validos).sum())
)

print(
    "Registros con identificadores repetidos:",
    int(duplicados_id.sum())
)

print(
    "Filas completamente duplicadas:",
    int(df_cen.duplicated().sum())
)


# ==========================================================
# 4. REVISAR ESTADOS ADMINISTRATIVOS
# ==========================================================

print("\n" + "=" * 65)
print("ESTADOS ADMINISTRATIVOS")
print("=" * 65)

estados = (
    df_cen["Estado"]
    .astype("string")
    .str.strip()
    .str.upper()
)

distribucion_estados = (
    estados
    .value_counts(dropna=False)
    .rename_axis("Estado")
    .reset_index(name="Registros")
)

display(distribucion_estados)


# ==========================================================
# 5. EVALUAR COMPLETITUD TERRITORIAL
# ==========================================================

print("\n" + "=" * 65)
print("CALIDAD DE LA INFORMACIÓN TERRITORIAL")
print("=" * 65)

variables_territoriales = [
    "Comuna",
    "Zona Afectada",
    "Consumo",
    "Distribuidoras Afectadas"
]

display(
    calidad_df[
        calidad_df["Variable"].isin(
            variables_territoriales
        )
    ].sort_values(
        "% faltantes",
        ascending=False
    )
)


# ==========================================================
# 6. EVALUAR COMPLETITUD TEMPORAL
# ==========================================================

print("\n" + "=" * 65)
print("CALIDAD DE LAS VARIABLES TEMPORALES")
print("=" * 65)

variables_temporales = [
    "Fecha Perturbación",
    "Fecha Efectiva Fin"
]

display(
    calidad_df[
        calidad_df["Variable"].isin(
            variables_temporales
        )
    ]
)


# ==========================================================
# 7. FINALIZAR DIAGNÓSTICO
# ==========================================================

print("\n" + "=" * 65)
print("DIAGNÓSTICO FINALIZADO CORRECTAMENTE")
print("=" * 65)

print(
    "\nLos datos originales permanecen sin modificaciones."
)

print(
    "Las decisiones de limpieza se definirán "
    "a partir de los resultados observados."
)

DIAGNÓSTICO GENERAL DE CALIDAD

Registros analizados: 18,019
Variables analizadas: 40

VARIABLES CON MAYOR PORCENTAJE DE DATOS FALTANTES


,Variable,Valores nulos,Campos vacíos,Total faltantes,% faltantes
7,Elemento(s),11460,0,11460,63.60
23,Distribuidoras Afectadas,11210,0,11210,62.21
6,SubEstación,9148,0,9148,50.77
26,Fecha Efectiva Fin,7214,0,7214,40.04
37,duracion_horas,7214,0,7214,40.04
12,Causa Definitiva,2153,0,2153,11.95
15,Causa Fenomeno Eléctrico,1462,361,1823,10.12
14,Causa Elementos,1312,359,1671,9.27
13,Causa Fenomeno Físico,1306,335,1641,9.11
9,Comuna,1145,0,1145,6.35



CONTROL DE IDENTIFICADORES

Identificadores ausentes o vacíos: 0
Registros con identificadores repetidos: 0
Filas completamente duplicadas: 0

ESTADOS ADMINISTRATIVOS


,Estado,Registros
0,FINALIZADO,10802
1,PENDIENTE,6138
2,ANULADO,1079



CALIDAD DE LA INFORMACIÓN TERRITORIAL


,Variable,Valores nulos,Campos vacíos,Total faltantes,% faltantes
23,Distribuidoras Afectadas,11210,0,11210,62.21
9,Comuna,1145,0,1145,6.35
8,Zona Afectada,1092,0,1092,6.06
22,Consumo,287,0,287,1.59



CALIDAD DE LAS VARIABLES TEMPORALES


,Variable,Valores nulos,Campos vacíos,Total faltantes,% faltantes
24,Fecha Perturbación,0,0,0,0.00
26,Fecha Efectiva Fin,7214,0,7214,40.04



DIAGNÓSTICO FINALIZADO CORRECTAMENTE

Los datos originales permanecen sin modificaciones.
Las decisiones de limpieza se definirán a partir de los resultados observados.


### 4.3. Resultados del diagnóstico de calidad

El diagnóstico se realizó sobre **18.019 registros y 40 variables** correspondientes al conjunto consolidado de informes de fallas del Sistema Eléctrico Nacional, provenientes de 33 publicaciones mensuales del Coordinador Eléctrico Nacional.

La evaluación permitió caracterizar la completitud, unicidad y disponibilidad de información técnica, territorial y temporal, identificando restricciones relevantes para las etapas posteriores de procesamiento.

#### 4.3.1. Completitud de la información

Los resultados evidencian diferencias importantes en la disponibilidad de información entre las variables analizadas.

| Variable                 | Registros sin información | Porcentaje de valores faltantes |
| ------------------------ | ------------------------: | ------------------------------: |
| Elemento(s)              |                    11.460 |                         63,60 % |
| Distribuidoras Afectadas |                    11.210 |                         62,21 % |
| SubEstación              |                     9.148 |                         50,77 % |
| Fecha Efectiva Fin       |                     7.214 |                         40,04 % |
| Causa Definitiva         |                     2.153 |                         11,95 % |
| Comuna                   |                     1.145 |                          6,35 % |
| Zona Afectada            |                     1.092 |                          6,06 % |
| Consumo                  |                       287 |                          1,59 % |
| Fecha Perturbación       |                         0 |                          0,00 % |

*Fuente: Elaboración propia a partir del diagnóstico de calidad del conjunto consolidado del CEN.*

La mayor proporción de información faltante se concentra en las variables de identificación de elementos, distribuidoras afectadas y subestaciones, con porcentajes superiores al 50 %.

Estas condiciones limitan la posibilidad de caracterizar exhaustivamente las instalaciones involucradas y establecer relaciones entre determinados elementos del sistema y los territorios registrados.

En particular, la variable `Distribuidoras Afectadas` presenta un 62,21 % de datos faltantes, por lo que su utilización como criterio único para determinar afectación territorial introduciría restricciones importantes de cobertura.

Por otra parte, la variable `Comuna` presenta un 6,35 % de valores faltantes. Esto significa que **16.874 informes disponen de información comunal**, equivalente al 93,65 % del conjunto documental.

Este resultado proporciona una base para desarrollar el análisis territorial, aunque la disponibilidad de un nombre comunal no garantiza por sí sola su correspondencia con una comuna oficial ni confirma interrupciones efectivas del suministro en ese territorio.

#### 4.3.2. Integridad y unicidad documental

La evaluación de identificadores presentó los siguientes resultados:

| Control de integridad                   | Resultado |
| --------------------------------------- | --------: |
| Informes analizados                     |    18.019 |
| Identificadores ausentes o vacíos       |         0 |
| Registros con identificadores repetidos |         0 |
| Filas completamente duplicadas          |         0 |

*Fuente: Elaboración propia mediante funciones de validación implementadas en Python.*

Los resultados confirman que, bajo los criterios de comparación aplicados, el conjunto consolidado mantiene la unicidad de los identificadores `id_falla` y no presenta filas completamente duplicadas.

Esta condición es relevante porque permite conservar una identificación documental individual para establecer posteriormente las relaciones entre informes y comunas.

Sin embargo, la ausencia de identificadores duplicados no demuestra necesariamente que cada informe corresponda a una falla física independiente. Esta distinción deberá mantenerse al construir e interpretar los indicadores territoriales.

#### 4.3.3. Estados administrativos

Los informes presentan tres estados administrativos:

| Estado     | Cantidad de informes |
| ---------- | -------------------: |
| Finalizado |               10.802 |
| Pendiente  |                6.138 |
| Anulado    |                1.079 |
| **Total**  |           **18.019** |

*Fuente: Elaboración propia a partir de la variable `Estado` del conjunto CEN.*

La distribución evidencia que los registros poseen diferentes condiciones administrativas, las cuales deben considerarse antes de construir indicadores de frecuencia, recurrencia y afectación operacional.

Los informes pendientes podrían contener antecedentes sujetos a actualización, mientras que los anulados requieren un tratamiento diferenciado para evitar interpretaciones incorrectas.

En consecuencia, durante el preprocesamiento se conservarán las categorías administrativas originales y se establecerán criterios explícitos para determinar qué registros participarán en cada análisis.

#### 4.3.4. Integridad temporal

La variable `Fecha Perturbación` presenta información en la totalidad de los registros, mientras que `Fecha Efectiva Fin` registra 7.214 valores faltantes, equivalentes al 40,04 %.

La disponibilidad de la fecha de perturbación permite estudiar inicialmente la distribución temporal de los informes.

No obstante, la ausencia de fechas efectivas de término restringe la posibilidad de calcular duraciones documentales para todos los registros.

Adicionalmente, la completitud de los campos no demuestra que las fechas sean cronológicamente correctas ni que sus zonas horarias hayan sido interpretadas adecuadamente. Estas condiciones deberán verificarse antes de efectuar cálculos de duración.

### 4.4. Implicancias del diagnóstico para el procesamiento

Los resultados obtenidos permiten establecer cuatro decisiones metodológicas:

1. **Conservar los 18.019 informes originales**, evitando eliminar registros únicamente por presentar información incompleta.
2. **Priorizar las variables comunales y de consumo**, considerando su importancia para el objetivo territorial del proyecto.
3. **Diferenciar los estados administrativos**, evitando contabilizar automáticamente informes pendientes o anulados como eventos confirmados de afectación.
4. **Validar las variables temporales**, antes de calcular duraciones o construir indicadores asociados a indisponibilidad operacional.

En consecuencia, el diagnóstico proporciona los antecedentes necesarios para implementar transformaciones controladas y documentadas, manteniendo la trazabilidad entre el conjunto original y los datos preparados.

Los resultados corresponden a una evaluación inicial de calidad y no constituyen todavía una validación definitiva de la afectación operacional sobre las comunas de Chile.


## 5. Limpieza y normalización de variables territoriales

### 5.1. Fundamentación técnica

El diagnóstico de calidad permitió identificar que el conjunto consolidado contiene 18.019 informes de fallas, de los cuales 16.874 presentan información en la variable `Comuna`, equivalente al 93,65 % de los registros.

Los 1.145 informes restantes no disponen de identificación comunal. Esta condición representa una restricción para el análisis territorial, pero no constituye evidencia suficiente para descartar su relevancia operacional.

Adicionalmente, se identificó que la variable `Distribuidoras Afectadas` presenta un 62,21 % de valores faltantes, mientras que `Consumo` registra un 1,59 %. Estas diferencias justifican mantener ambas variables como antecedentes complementarios, sin asumir que su ausencia representa inexistencia de afectación.

A partir de estos resultados se establece un procedimiento de normalización destinado a estandarizar la representación de las variables territoriales y facilitar su posterior integración con los registros oficiales del Instituto Nacional de Estadísticas (INE).

### 5.2. Criterios de transformación

El procesamiento considera las siguientes decisiones:

**Conservación de la información original.** Se mantendrán las 18.019 observaciones iniciales, sin eliminar informes por ausencia de comuna, estado administrativo o información de consumo.

**Normalización de texto.** Se eliminarán espacios innecesarios y se estandarizará la representación de determinadas categorías, conservando los valores originales para garantizar trazabilidad.

**Tratamiento de múltiples comunas.** Los campos que contengan varias comunas separadas por el carácter `|` se descompondrán en registros individuales de asociación territorial.

**Identificación territorial.** Se generará una clave de comparación textual para cada comuna, utilizando un criterio uniforme de mayúsculas, espacios y caracteres acentuados. Esta clave no sustituirá al código comunal oficial.

**Control de duplicidad territorial.** Se evitará que una misma comuna aparezca repetida dentro de las asociaciones de un mismo informe.

**Separación de unidades de análisis.** Se distinguirá entre la unidad documental, correspondiente al informe de falla, y la unidad de asociación territorial, correspondiente a cada combinación entre informe y comuna.

### 5.3. Estructura de datos resultante

La transformación generará dos estructuras complementarias:

**Base documental:** conservará los informes originales y agregará variables normalizadas para el procesamiento posterior.

**Base de asociaciones territoriales:** contendrá las relaciones identificadas entre informes y comunas, manteniendo el identificador documental y el nombre territorial reportado.

Esta estructura permitirá examinar la recurrencia territorial sin multiplicar artificialmente el número de informes de fallas.

La normalización textual no garantiza todavía que todos los nombres correspondan a comunas oficiales de Chile. La homologación territorial se realizará posteriormente mediante el maestro de comunas del INE.

Asimismo, la presencia de una comuna en un informe no permitirá concluir que todos sus habitantes o clientes experimentaron una interrupción del suministro eléctrico.


In [6]:
# ==========================================================
# FASE 2 - APARTADO 5
# LIMPIEZA Y NORMALIZACIÓN TERRITORIAL
# PROYECTO MCDI500 - GRUPO 7
# ==========================================================

import pandas as pd
import re
import unicodedata

from IPython.display import display


# ==========================================================
# 1. FUNCIÓN PARA LIMPIAR TEXTO
# ==========================================================

def limpiar_texto(valor):
    """
    Elimina espacios innecesarios.

    Conserva los valores ausentes como NA.
    No modifica el significado del contenido.
    """

    if pd.isna(valor):
        return pd.NA

    texto = str(valor)

    texto = re.sub(r"\s+", " ", texto).strip()

    if texto == "":
        return pd.NA

    return texto


# ==========================================================
# 2. FUNCIÓN PARA NORMALIZAR CATEGORÍAS
# ==========================================================

def normalizar_categoria(valor):
    """
    Limpia una categoría y la convierte a mayúsculas.
    """

    texto = limpiar_texto(valor)

    if pd.isna(texto):
        return pd.NA

    return texto.upper()


# ==========================================================
# 3. FUNCIÓN PARA CREAR CLAVES COMUNALES
# ==========================================================

def normalizar_comuna(valor):
    """
    Genera una clave textual de comparación.

    - Elimina diferencias entre mayúsculas y minúsculas.
    - Elimina diferencias de acentuación.
    - Normaliza espacios.

    No asigna códigos oficiales de comuna.
    """

    texto = limpiar_texto(valor)

    if pd.isna(texto):
        return pd.NA

    texto = unicodedata.normalize("NFKD", texto)

    texto = "".join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
    )

    return texto.upper()


# ==========================================================
# 4. CREAR COPIA DEL DATASET ORIGINAL
# ==========================================================

df_f2 = df_cen.copy(deep=True)

registros_originales = len(df_cen)


# ==========================================================
# 5. NORMALIZAR VARIABLES
# ==========================================================

df_f2["comuna_limpia"] = (
    df_f2["Comuna"]
    .map(limpiar_texto)
    .astype("string")
)

df_f2["estado_normalizado"] = (
    df_f2["Estado"]
    .map(normalizar_categoria)
    .astype("string")
)

df_f2["consumo_normalizado"] = (
    df_f2["Consumo"]
    .map(normalizar_categoria)
    .astype("string")
)

df_f2["zona_afectada_limpia"] = (
    df_f2["Zona Afectada"]
    .map(limpiar_texto)
    .astype("string")
)


# ==========================================================
# 6. IDENTIFICAR Y SEPARAR COMUNAS
# ==========================================================

asociaciones = []

for id_falla, comunas, zona in df_f2[
    ["id_falla", "comuna_limpia", "zona_afectada_limpia"]
].itertuples(index=False, name=None):

    if pd.isna(comunas):
        continue

    # El delimitador identificado en los datos es "|".
    lista_comunas = str(comunas).split("|")

    for comuna in lista_comunas:

        comuna_limpia = limpiar_texto(comuna)

        if pd.isna(comuna_limpia):
            continue

        asociaciones.append({
            "id_falla": id_falla,
            "comuna_reportada": comuna_limpia,
            "comuna_clave": normalizar_comuna(
                comuna_limpia
            ),
            "zona_afectada": zona
        })


# ==========================================================
# 7. CREAR BASE DE ASOCIACIONES TERRITORIALES
# ==========================================================

df_comunas = pd.DataFrame(
    asociaciones,
    columns=[
        "id_falla",
        "comuna_reportada",
        "comuna_clave",
        "zona_afectada"
    ]
)

asociaciones_iniciales = len(df_comunas)


# ==========================================================
# 8. EVITAR ASOCIACIONES REPETIDAS
# ==========================================================

df_comunas = df_comunas.drop_duplicates(
    subset=["id_falla", "comuna_clave"]
).reset_index(drop=True)

asociaciones_duplicadas = (
    asociaciones_iniciales - len(df_comunas)
)


# ==========================================================
# 9. VALIDACIONES DE INTEGRIDAD
# ==========================================================

# Verificar que no se eliminaron informes originales.
assert len(df_f2) == registros_originales, (
    "Se modificó la cantidad de informes originales."
)

# Verificar que todas las columnas originales se conservaron.
pd.testing.assert_frame_equal(
    df_f2[df_cen.columns],
    df_cen
)

# Verificar que las asociaciones territoriales sean únicas.
assert not df_comunas.duplicated(
    subset=["id_falla", "comuna_clave"]
).any(), (
    "Existen asociaciones territoriales duplicadas."
)

# Verificar que no se generaron claves vacías.
assert df_comunas["comuna_clave"].notna().all(), (
    "Se generaron claves comunales nulas."
)


# ==========================================================
# 10. RESULTADOS DEL PROCESAMIENTO
# ==========================================================

print("=" * 65)
print("NORMALIZACIÓN TERRITORIAL - FASE 2")
print("=" * 65)

print("\nBASE DOCUMENTAL")

print("Informes originales:", registros_originales)
print("Informes conservados:", len(df_f2))

print(
    "Informes sin comuna informada:",
    int(df_f2["comuna_limpia"].isna().sum())
)

print(
    "Informes con comuna informada:",
    int(df_f2["comuna_limpia"].notna().sum())
)


print("\nASOCIACIONES TERRITORIALES")

print(
    "Asociaciones generadas:",
    len(df_comunas)
)

print(
    "Informes asociados a alguna comuna:",
    df_comunas["id_falla"].nunique()
)

print(
    "Asociaciones repetidas detectadas:",
    asociaciones_duplicadas
)

print(
    "Claves comunales distintas:",
    df_comunas["comuna_clave"].nunique()
)


# ==========================================================
# 11. IDENTIFICAR INFORMES MULTICOMUNA
# ==========================================================

informes_multicomuna = (
    df_f2["comuna_limpia"]
    .fillna("")
    .str.contains("|", regex=False)
    .sum()
)

print(
    "\nInformes con delimitador multicomuna:",
    int(informes_multicomuna)
)


# ==========================================================
# 12. MOSTRAR EJEMPLOS DE LA TRANSFORMACIÓN
# ==========================================================

print("\nEJEMPLOS DE ASOCIACIONES TERRITORIALES")

display(
    df_comunas.head(10)
)


# ==========================================================
# 13. VERIFICAR NORMALIZACIÓN DE ESTADOS
# ==========================================================

print("\nESTADOS ADMINISTRATIVOS NORMALIZADOS")

display(
    df_f2["estado_normalizado"]
    .value_counts(dropna=False)
    .rename_axis("Estado")
    .reset_index(name="Registros")
)


# ==========================================================
# 14. VERIFICAR NORMALIZACIÓN DE CONSUMO
# ==========================================================

print("\nCATEGORÍAS DE CONSUMO NORMALIZADAS")

display(
    df_f2["consumo_normalizado"]
    .value_counts(dropna=False)
    .rename_axis("Consumo")
    .reset_index(name="Registros")
)


# ==========================================================
# 15. FINALIZAR
# ==========================================================

print("\n" + "=" * 65)

print("NORMALIZACIÓN FINALIZADA CORRECTAMENTE")

print("[OK] Se conservaron todos los informes originales.")
print("[OK] Se construyó la relación informe-comuna.")
print("[OK] No existen asociaciones territoriales duplicadas.")

print(
    "[PENDIENTE] Homologar comunas con códigos oficiales INE."
)

print("=" * 65)

NORMALIZACIÓN TERRITORIAL - FASE 2

BASE DOCUMENTAL
Informes originales: 18019
Informes conservados: 18019
Informes sin comuna informada: 1145
Informes con comuna informada: 16874

ASOCIACIONES TERRITORIALES
Asociaciones generadas: 23284
Informes asociados a alguna comuna: 16874
Asociaciones repetidas detectadas: 0
Claves comunales distintas: 320

Informes con delimitador multicomuna: 2623

EJEMPLOS DE ASOCIACIONES TERRITORIALES


,id_falla,comuna_reportada,comuna_clave,zona_afectada
0,2024000001,San Clemente,SAN CLEMENTE,Maule
1,2024000002,Huara,HUARA,Tarapacá
2,2024000004,Camarones,CAMARONES,Arica y Parinacota
3,2024000003,Camarones,CAMARONES,Arica y Parinacota
4,2024000087,Los Andes,LOS ANDES,Valparaíso
5,2024000088,Sierra Gorda,SIERRA GORDA,Antofagasta
6,2024000089,Calama,CALAMA,Antofagasta
7,2024000090,Calama,CALAMA,Antofagasta
8,2024000091,Calama,CALAMA,Antofagasta
9,2024000092,Copiapó,COPIAPO,Atacama



ESTADOS ADMINISTRATIVOS NORMALIZADOS


,Estado,Registros
0,FINALIZADO,10802
1,PENDIENTE,6138
2,ANULADO,1079



CATEGORÍAS DE CONSUMO NORMALIZADAS


,Consumo,Registros
0,CONSUMO_NO_AFECTADO,10901
1,CONSUMO_REGULADO,3932
2,CONSUMO_LIBRE_REGULADO,1878
3,CONSUMO_LIBRE,1021
4,<NA>,287



NORMALIZACIÓN FINALIZADA CORRECTAMENTE
[OK] Se conservaron todos los informes originales.
[OK] Se construyó la relación informe-comuna.
[OK] No existen asociaciones territoriales duplicadas.
[PENDIENTE] Homologar comunas con códigos oficiales INE.


### 5.4. Resultados de la normalización territorial

El procedimiento de limpieza y normalización se ejecutó sobre los 18.019 registros consolidados del Coordinador Eléctrico Nacional, generando variables estandarizadas y una estructura de asociaciones territoriales entre informes de fallas y comunas mencionadas.

La transformación se desarrolló conservando las columnas originales, sin eliminar observaciones y manteniendo la identificación documental de cada informe.

#### 5.4.1. Conservación e integridad de los registros

Los resultados de las verificaciones fueron los siguientes:

| Indicador                                         | Resultado |
| ------------------------------------------------- | --------: |
| Informes originales                               |    18.019 |
| Informes conservados después de la transformación |    18.019 |
| Informes con comuna informada                     |    16.874 |
| Informes sin comuna informada                     |     1.145 |
| Asociaciones territoriales generadas              |    23.284 |
| Informes asociados a una o más comunas            |    16.874 |
| Informes con delimitador multicomuna              |     2.623 |
| Asociaciones repetidas detectadas                 |         0 |
| Claves comunales distintas                        |       320 |

*Fuente: Elaboración propia mediante el procedimiento de normalización territorial implementado en Python.*

Los resultados confirman que el procesamiento conservó la totalidad de los informes originales y permitió construir una segunda estructura de datos orientada al análisis territorial.

La base resultante contiene **23.284 asociaciones informe–comuna**, correspondientes a 16.874 informes que presentan información comunal.

La diferencia entre ambas cantidades se explica porque un mismo informe puede mencionar más de una comuna.

#### 5.4.2. Caracterización de informes multicomuna

Se identificaron **2.623 informes que contienen el delimitador `|`**, utilizado para representar varias comunas dentro de un mismo campo.

El procedimiento permitió separar estas denominaciones y generar asociaciones territoriales individuales, conservando el identificador `id_falla` como vínculo con el informe de origen.

Por ejemplo, un registro que mencione las comunas de Concepción, Chiguayante, Talcahuano y Hualpén se representa mediante cuatro asociaciones territoriales vinculadas a un mismo informe.

Esta transformación modifica la estructura de representación territorial, pero no incrementa el número de informes originales.

La distinción resulta fundamental para evitar interpretaciones incorrectas de frecuencia, puesto que **23.284 asociaciones territoriales no equivalen a 23.284 fallas independientes del Sistema Eléctrico Nacional**.

Asimismo, la prueba de unicidad no detectó asociaciones repetidas para una misma combinación de `id_falla` y `comuna_clave`.

#### 5.4.3. Normalización de denominaciones comunales

El procedimiento incorporó una clave textual estandarizada mediante normalización de espacios, mayúsculas y caracteres acentuados.

Por ejemplo:

| Comuna reportada | Clave de comparación |
| ---------------- | -------------------- |
| San Clemente     | SAN CLEMENTE         |
| Huara            | HUARA                |
| Los Andes        | LOS ANDES            |
| Sierra Gorda     | SIERRA GORDA         |
| Calama           | CALAMA               |
| Copiapó          | COPIAPO              |

*Fuente: Elaboración propia a partir de las transformaciones realizadas sobre los registros del CEN.*

Como resultado, se identificaron **320 claves comunales distintas**.

Este valor representa las denominaciones normalizadas presentes en los registros analizados y no debe interpretarse todavía como una cantidad de comunas oficialmente homologadas.

La validación definitiva requerirá comparar estas claves con el catálogo territorial oficial del Instituto Nacional de Estadísticas, verificando nombres, códigos comunales y posibles diferencias de denominación.

#### 5.4.4. Caracterización de las categorías de consumo

La normalización de la variable `Consumo` permitió identificar cinco condiciones:

| Categoría              |   Informes |
| ---------------------- | ---------: |
| CONSUMO_NO_AFECTADO    |     10.901 |
| CONSUMO_REGULADO       |      3.932 |
| CONSUMO_LIBRE_REGULADO |      1.878 |
| CONSUMO_LIBRE          |      1.021 |
| Sin información        |        287 |
| **Total**              | **18.019** |

*Fuente: Elaboración propia a partir de la variable `Consumo` del conjunto CEN.*

Los resultados muestran que **6.831 informes presentan categorías de consumo distintas de `CONSUMO_NO_AFECTADO`**, mientras que 10.901 se encuentran clasificados explícitamente como consumo no afectado.

Otros 287 informes no disponen de información en esta variable.

Esta clasificación permite identificar antecedentes documentales potencialmente relevantes para estudiar afectación del consumo eléctrico.

Sin embargo, **la categoría de consumo no constituye por sí sola una medida cuantitativa de indisponibilidad operacional, duración de interrupciones ni cantidad de clientes afectados**.

Por esta razón, las categorías normalizadas se conservarán como variables descriptivas y de clasificación, sujetas a validación adicional antes de utilizarlas en indicadores de impacto territorial.

### 5.5. Evaluación de la transformación y continuidad del análisis

La ejecución del procedimiento permitió verificar los siguientes aspectos:

* Se conservaron los 18.019 informes originales y sus columnas iniciales.
* Se generaron 23.284 asociaciones territoriales sin modificar la cantidad de informes.
* Se identificaron 2.623 informes con delimitador multicomuna.
* Se obtuvieron 320 claves territoriales normalizadas.
* No se detectaron asociaciones repetidas bajo la combinación de identificador documental y clave comunal.
* Se estandarizaron las categorías administrativas y de consumo, conservando la información de origen.

Estos resultados permiten considerar finalizada la etapa de **normalización textual y separación inicial de asociaciones territoriales**.

No obstante, la preparación de los datos aún requiere validar la correspondencia geográfica de las comunas, revisar posibles denominaciones no reconocidas oficialmente y vincular los registros con información demográfica territorial.

En la siguiente etapa se utilizará información oficial del Instituto Nacional de Estadísticas para construir un maestro de comunas que permita establecer correspondencias verificables y seleccionar las 15 comunas prioritarias según la población censada en 2024.

La integración se desarrollará manteniendo la distinción entre informes documentales, asociaciones territoriales y afectación efectiva del suministro eléctrico, evitando atribuir impactos operacionales que no estén respaldados por los antecedentes disponibles.


## 6. Integración y homologación territorial con el Instituto Nacional de Estadísticas

### 6.1. Fundamentación técnica

El procesamiento inicial permitió conservar los 18.019 informes originales y construir una estructura de 23.284 asociaciones territoriales, correspondientes a 16.874 informes con información comunal disponible.

La normalización identificó 320 claves comunales distintas. Sin embargo, estas claves representan denominaciones extraídas de los informes del Coordinador Eléctrico Nacional y todavía no constituyen identificadores territoriales oficialmente validados.

Para garantizar la consistencia geográfica del análisis, se incorporará información del Censo de Población y Vivienda 2024, publicado por el Instituto Nacional de Estadísticas (INE).

Esta fuente permitirá establecer un marco territorial de referencia para verificar las comunas mencionadas en los informes y seleccionar aquellas que presentan mayor población censada.

### 6.2. Criterios de integración

La integración territorial considerará los siguientes criterios:

**Identificación oficial:** utilizar los nombres y códigos comunales oficiales disponibles en la información territorial de referencia.

**Homologación:** comparar las denominaciones reportadas por el CEN con los nombres oficiales, identificando coincidencias, diferencias y registros que requieren revisión.

**Consistencia geográfica:** utilizar antecedentes regionales para resolver posibles ambigüedades, cuando la información disponible lo permita.

**Priorización territorial:** seleccionar las 15 comunas de mayor población censada según el Censo 2024, mediante un procedimiento reproducible.

**Trazabilidad:** conservar los nombres originales del CEN y registrar las reglas utilizadas para establecer cada correspondencia territorial.

### 6.3. Restricciones metodológicas

La población censada se utilizará como criterio de priorización y caracterización territorial.

No se interpretará como número de clientes eléctricos ni como cantidad de habitantes efectivamente afectados por una interrupción.

La homologación comunal tampoco permitirá afirmar, por sí sola, que todos los sectores de una comuna mencionada en un informe experimentaron indisponibilidad de suministro.

Los registros que no puedan asociarse de manera verificable con una comuna oficial permanecerán identificados como pendientes de homologación, evitando asignaciones arbitrarias.

### 6.4. Procedimiento

El proceso comenzará con la obtención y revisión de los tabulados oficiales del INE.

Posteriormente, se identificarán las variables necesarias para construir el maestro territorial y se establecerán las correspondencias con las claves comunales normalizadas del CEN.

Finalmente, se verificará la integridad de la integración y se seleccionarán las comunas prioritarias del estudio.


In [8]:
# ==========================================================
# FASE 2 - APARTADO 6
# RECONOCIMIENTO DE LA INFORMACIÓN TERRITORIAL DEL INE
# ==========================================================

import pandas as pd
from IPython.display import display

# 1. Identificar el archivo oficial del INE

ARCHIVO_INE = (
    RAIZ
    / "data"
    / "external"
    / "ine_censo2024_poblacion_comunal.xlsx"
)

# 2. Verificar que el archivo exista

if not ARCHIVO_INE.is_file():
    raise FileNotFoundError(
        f"No se encontró el archivo del INE: {ARCHIVO_INE}"
    )

# 3. Abrir el archivo Excel

libro_ine = pd.ExcelFile(ARCHIVO_INE)

# 4. Identificar las hojas disponibles

print("=" * 60)
print("EXPLORACIÓN DEL ARCHIVO OFICIAL DEL INE")
print("=" * 60)

print("\nArchivo:", ARCHIVO_INE.name)

print("\nHOJAS DISPONIBLES:")

for numero, hoja in enumerate(libro_ine.sheet_names, start=1):
    print(f"{numero}. {hoja}")

# 5. Mostrar las primeras filas de las hojas

for hoja in libro_ine.sheet_names[:3]:

    print("\n" + "=" * 60)
    print("HOJA:", hoja)
    print("=" * 60)

    muestra = pd.read_excel(
        libro_ine,
        sheet_name=hoja,
        header=None,
        nrows=12
    )

    display(muestra)

print("\n[OK] Reconocimiento del archivo INE finalizado.")

EXPLORACIÓN DEL ARCHIVO OFICIAL DEL INE

Archivo: ine_censo2024_poblacion_comunal.xlsx

HOJAS DISPONIBLES:
1. Nota
2. 1
3. 2
4. 3
5. 4

HOJA: Nota


,0,1,2,3,4
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,Nota de presentación
2,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,"Fecha de publicación: 27 de marzo, 2025.",NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN
6,NaN,1. Introducción,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN
8,NaN,\nEn el contexto de la primera publicación de ...,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN



HOJA: 1


,0,1,2,3,4,5
0,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,1. Población censada por sexo y razón hombre-m...,NaN,NaN,NaN,NaN,NaN
3,Código región,Región,Población censada,Hombres,Mujeres,Razón hombre-mujer
4,0,País,18480432,8967033,9513399,94.3
5,15,Arica y Parinacota,244569,120381,124188,96.9
6,1,Tarapacá,369806,183343,186463,98.3
7,2,Antofagasta,635416,313995,321421,97.7
8,3,Atacama,299180,148263,150917,98.2
9,4,Coquimbo,832864,404397,428467,94.4



HOJA: 2


,0,1,2,3,4,5,6,7,8,9
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2. Población censada por sexo y razón hombre-m...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Código región,Región,Código provincia,Provincia,Código comuna,Comuna,Población censada,Hombres,Mujeres,Razón hombre-mujer
4,0,País,0,País,0,País,18480432,8967033,9513399,94.3
5,15,Arica y Parinacota,151,Arica,15101,Arica,241653,118844,122809,96.8
6,15,Arica y Parinacota,151,Arica,15102,Camarones,861,430,431,99.8
7,15,Arica y Parinacota,152,Parinacota,15201,Putre,1547,846,701,120.7
8,15,Arica y Parinacota,152,Parinacota,15202,General Lagos,508,261,247,105.7
9,1,Tarapacá,11,Iquique,1101,Iquique,199587,98413,101174,97.3



[OK] Reconocimiento del archivo INE finalizado.
